In [0]:
from pyspark.sql import functions as F

PATH_REFERENCES = "/Volumes/aviation/bronze/archives/references/"

# A character which doesn't exist in the file -> disables the quoting of the CSV reader
NO_QUOTES = chr(0)

In [0]:
df_airfield = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("encoding", "ISO-8859-1")
    .option("quote", NO_QUOTES)
    .load(f"{PATH_REFERENCES}/AerodromosPublicos.csv")
)

In [0]:
df_airfield = df_airfield.select(
    F.col("`Código OACI`").alias("icao"),
    F.col("CIAD").alias("ciad"),
    F.col("Nome").alias("nome"),
    F.col("`Município`").alias("municipio"),
    F.col("UF").alias("uf"),
    F.col("`Município Servido`").alias("municipio_servido"),
    F.col("`UF Servido`").alias("uf_servido"),
    F.col("Latitude").alias("latitude"),
    F.col("Longitude").alias("longitude"),
    F.col("Altitude").alias("altitude"),
    F.col("`Situação`").alias("situacao"),
).withColumn("_ingerido_em", F.current_timestamp())

df_airfield.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("aviation.bronze.airfield")

print(f"bronze.airfield: {spark.table('aviation.bronze.airfield').count():,} linhas")
display(spark.sql("SELECT icao, nome, municipio, uf FROM aviation.bronze.airfield WHERE icao IN ('SBRB','SBGR','SBSP','SBFZ')"))


In [0]:
def read_companies(arquivo: str):
    """Le um cadastro de empresas. Sem uniao, sem enriquecimento: uma tabela por arquivo."""
    return (
        spark.read.format("csv")
        .option("sep", ";")
        .option("header", "true")
        .option("skipRows", 1)
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .load(f"{PATH_REFERENCES}/{arquivo}")
        .select(
            F.col("ICAO").alias("icao"),
            F.col("Estrangeira").alias("sigla_iata"),
            F.col("Razao").alias("razao_social"),
            F.col("Servico").alias("servico"),
            F.col("Cidade").alias("cidade"),
            F.col("UF").alias("uf"),
            F.col("Ativa").alias("situacao"),
        )
        .withColumn("_arquivo_origem", F.lit(arquivo))
        .withColumn("_ingerido_em", F.current_timestamp())
    )

In [0]:
for archive, schema in [
    ("pda_empresas_aereas_nacionais.csv",    "aviation.bronze.national_companies"),
    ("pda_empresas_aereas_estrangeiros.csv", "aviation.bronze.foreign_companies"),
]:
    read_companies(archive).write.format("delta").mode("overwrite").option(
        "overwriteSchema", "true"
    ).saveAsTable(schema)
    print(f"{schema}: {spark.table(schema).count():,} linhas")

In [0]:
display(spark.sql("""
    SELECT 'national_companies' AS tabela, COUNT(*) AS linhas,
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao
    FROM aviation.bronze.national_companies
    UNION ALL
    SELECT 'foreign_companies', COUNT(*),
           COUNT(CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END)
    FROM aviation.bronze.foreign_companies
"""))

In [0]:
display(spark.sql("""
    SELECT icao, razao_social, servico, uf, situacao
    FROM aviation.bronze.national_companies
    WHERE icao IN ('GLO','TAM','AZU','PAM')
    ORDER BY icao
"""))

display(spark.sql("""
    SELECT icao, razao_social, servico, situacao
    FROM aviation.bronze.foreign_companies
    WHERE icao IN ('AAL','TAP','AVA','ARG')
    ORDER BY icao
"""))

In [0]:
CODES = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira"),
]

codes = spark.createDataFrame(CODES, "domain string, code string, description string")
codes.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("aviation.bronze.operation_codes")

print(f"bronze.operation_codes: {spark.table('aviation.bronze.operation_codes').count()} linhas")
display(spark.table("aviation.bronze.operation_codes"))

In [0]:
display(spark.sql("SHOW TABLES IN aviation.bronze"))

In [0]:
display(spark.sql("""
    SELECT version, timestamp, operation,
           operationMetrics.numOutputRows AS linhas_escritas
    FROM (DESCRIBE HISTORY aviation.bronze.vra)
    ORDER BY version
"""))

In [0]:
display(spark.sql("""
    SELECT 'versao 0 (1a carga)'   AS versao,
           COUNT(*)                AS linhas,
           MIN(_ingerido_em)       AS ingerido_em
    FROM aviation.bronze.vra VERSION AS OF 0
    UNION ALL
    SELECT 'versao atual', COUNT(*), MIN(_ingerido_em)
    FROM aviation.bronze.vra
"""))